# Barranquilla MOPAC: una reacción enzimática en la GPU

Primer paso de la acilación en una serina proteasa: el oxígeno OG de **Ser195** de la tripsina ataca el carbono
carbonílico de **Lys15** de BPTI (PDB 2PTC; el enlace Lys15–Ala16 ocupa el sitio activo como sustrato) mientras
**His57** toma el protón de Ser195. `scripts/qmmm/qmmm_scan.py` hace un barrido QM/MM (mapeo adiabático) sobre dos
coordenadas a la vez: la distancia Ser195 OG – Lys15 C y la transferencia del protón, d(OG–HG) − d(HG–NE2 de His57)
(−1,1 Å: el protón en Ser195; +1 Å: en His57). Prepara el sistema como `run_qmmm_md.py` (hidrógenos, amber14 + TIP3P,
caja de agua, equilibrado MM y minimización MM de todo el sistema), minimiza el reactivo con QM/MM y recorre una malla
de las dos coordenadas con restricciones armónicas, fila por fila; en cada punto minimiza el sitio activo (los residuos
a menos de 10 Å de la región QM; el resto queda fijo donde lo dejó la minimización MM) partiendo del punto anterior.
Al final vuelve al reactivo por el borde de la malla (última fila y primera columna), y la diferencia entre ida y
vuelta en los mismos puntos mide cuánto depende el mapa del camino. Por último (`RELEASE`) quita las restricciones y
minimiza desde cada punto de la última fila y desde cada punto de la malla más bajo que todos sus vecinos: si la
estructura se queda donde la dejó el barrido, está en un mínimo; si se va, no. `HESSIAN` comprueba cada una de esas
estructuras con las frecuencias de los átomos QM (el resto fijo): un mínimo no tiene frecuencias imaginarias. Las
energías se miden desde el mínimo más bajo de la región del reactivo (protón en Ser195, OG···C > 2 Å), no desde la
estructura de partida.

Cada minimización sigue hasta que la fuerza más grande sobre un átomo que se mueve baja de `FMAX` (10 kJ/mol/nm =
0,24 kcal/mol/Å); los átomos a más de 10 Å de la región QM quedan fijos. Las minimizaciones QM/MM usan un L-BFGS propio
en el que ningún átomo se mueve más de `MAX_STEP` (0,1 Å) por iteración: el minimizador de OpenMM empieza cada llamada
con un paso de 1 nm en todas las coordenadas juntas, y con él la primera prueba (commit e78777a2, en una L4) se detuvo
en un SCF que no convergió. Si aun así un SCF no converge, se abre de nuevo la sesión de MOPAC en la última estructura
aceptada y se acorta el paso. Cada paso se acepta por las fuerzas, no por la energía: con una búsqueda de línea que
pedía que la energía bajara (commit 552409e4, una T4), 532 de 766 minimizaciones se detuvieron sin converger, porque la
energía que queda por bajar con 20 kJ/mol/nm (~5·10⁻⁴ kJ/mol en un enlace con hidrógeno) es menor que el ruido de la
energía MM (~10⁻³ kJ/mol, medido en CPU). Con las fuerzas (commit 9a0c007a, una A100) convergieron 728 de las 740
minimizaciones de cinco réplicas, ningún SCF falló, y la parte MOPAC de la energía concordó con sus fuerzas (dentro de
un 6,1 %) para desplazamientos de 10⁻⁵ y 10⁻⁴ nm en los tres puntos que midió `--probe`. Las 12 que no convergieron se
detuvieron porque la búsqueda de línea no encontró ningún paso aceptable. Cada intento empieza su SCF desde los
orbitales del anterior, así que tras un intento largo el SCF puede quedar en otra solución: ahora la búsqueda se repite
una vez más con la energía y las fuerzas de la misma estructura calculadas con un SCF empezado de nuevo, y `scan.log`
guarda los intentos de cada búsqueda sin paso y la energía de esa estructura otra vez y desde cero (así se ve si el SCF
cambió de solución).
La prueba corta mide, con `--probe`, si la energía concuerda con las fuerzas (parte MOPAC y parte OpenMM por separado)
en el reactivo y en los puntos que no convergen, y el cuaderno solo sigue con las réplicas si converge al menos el 80 %
de sus puntos. El resumen de cada réplica dice cuántos puntos no convergieron, la fuerza que quedó, cuántos SCF hubo que
empezar de nuevo y cuántas búsquedas de línea no encontraron paso.

Todas las réplicas preparan el mismo sistema: OpenMM coloca los hidrógenos y los iones con números aleatorios y
minimiza los hidrógenos en la plataforma por defecto, que en la GPU no es determinista (en 9a0c007a la prueba y la
réplica 1, con la misma semilla, tenían 46 583 y 46 658 átomos); ahora los números aleatorios son los mismos en cada
corrida y los hidrógenos se minimizan en la plataforma Reference de OpenMM. Con la misma semilla, la GPU repite además el mismo equilibrado MM (fuerzas deterministas de OpenMM), así que
la prueba y la réplica 1 deben partir del mismo reactivo; el resumen lo comprueba. Las réplicas corren a la vez en la
GPU. Con una sola coordenada
(OG–C) y el protón libre, un barrido en CPU dio una subida monótona de la energía sin que el protón pasara a His57:
hacen falta las dos.
Región QM: cadenas laterales de His57 (HID), Asp102 y Ser195 y los residuos Lys15–Ala16 de BPTI (54 átomos + 5 de
enlace), PM7/MOZYME en la GPU; la electrostática QM–MM de largo alcance por PME.

| Bloque | Qué hace | Tiempo (aprox.) |
|---|---|---|
| Compilar | MOPAC con la GPU | ~10 min |
| Prueba | una réplica corta: malla de 3 × 3 puntos, vuelta, minimizaciones sin restricciones y frecuencias | con 9a0c007a, 1 min de minimización en una A100 |
| Mapas | por semilla (equilibrado MM propio): una malla de 120–132 puntos (según la distancia OG···C del reactivo), 9–21 de vuelta y una minimización sin restricciones, con sus frecuencias, por punto de la última fila y por mínimo local de la malla | con 9a0c007a, 24–28 min de minimización por réplica, las cinco a la vez en una A100 (las frecuencias aparte) |
| Figuras | el mapa de energía de cada réplica | segundos |

**Qué es y qué no es:** cada punto es una estructura minimizada; el mapa es de energía potencial, no de energía
libre (sin muestreo, entropía ni energía de punto cero), y depende del orden en que se recorre la malla y de la
estructura de partida: por eso hay cinco réplicas y la vuelta.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (una L4 basta; A100 o T4 también
sirven), y luego *Ejecutar todo*. Primero corre una réplica corta (`TEST_FIRST`): si falla, el cuaderno se detiene
antes de las cinco réplicas. Los resultados se guardan en Google Drive (`MyDrive/mopac_gpu_runs/RESULTS_NAME`); si la
sesión se corta, vuelva a ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '3b1b0e3b2c6b11cac5a42644d4ed7b64a8696ba5'           # qmmm_scan.py
SEEDS = [1, 2, 3, 4, 5]        # una réplica por semilla (equilibrado MM y velocidades propios)
PARALLEL = True                # todas las réplicas a la vez en la GPU
SCAN = 'E:195:OG-I:15:C'       # coordenada 1: Ser195 OG – Lys15 C (BPTI)
TO, STEP = 1.45, 0.1           # Å: desde el reactivo hasta TO
SCAN2 = 'E:195:OG-E:195:HG-E:57:NE2'   # coordenada 2: d(OG–HG) − d(HG–NE2), transferencia del protón
FROM2, TO2, STEP2 = -1.1, 1.1, 0.2
FMAX = 10.0                    # kJ/mol/nm: convergida cuando la fuerza más grande sobre un átomo móvil baja de esto
MAX_STEP = 0.1                 # Å: lo más que se mueve un átomo en una iteración de la minimización QM/MM
MIN_ITER, MIN_ITER_START = 5000, 20000   # tope de iteraciones por punto y para el reactivo
BACK = True                    # volver al reactivo por el borde de la malla (histéresis)
RELEASE, RELEASE_ITER = True, 20000  # sin restricciones desde la última fila y desde los mínimos de la malla
HESSIAN = True                 # frecuencias de los átomos QM en cada estructura sin restricciones
MM_EQUIL_PS = 50
TOLERANCE = 0.001              # criterio SCF relativo de MOPAC
USE_DRIVE = True
TEST_FIRST = True              # antes, una réplica corta (semilla 1, malla gruesa): prueba el código y la reproducibilidad
PROBE = 4                      # en la prueba: energía contra fuerzas en el reactivo y en hasta 4 puntos sin converger
MIN_CONVERGED = 0.8            # fracción de puntos de la prueba que deben converger para seguir con las réplicas
REQUIRE_GPU = None             # 'A100' para exigirla; None: cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import csv, os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno '
                       f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')
tag = re.sub(r'[^a-z0-9]+', '', gpu.split(',')[0].lower().replace('nvidia', ''))[:12]
RESULTS_NAME = f'enzyme_scan_{tag}_{COMMIT[:8]}'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_gpu_runs')
else:
    BASE = Path('/content/mopac_gpu_runs')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT, 'seeds': SEEDS,
                                          'scan': SCAN, 'to': TO, 'step': STEP, 'scan2': SCAN2, 'from2': FROM2,
                                          'to2': TO2, 'step2': STEP2, 'min_iter': MIN_ITER,
                                          'min_iter_start': MIN_ITER_START, 'mm_equil_ps': MM_EQUIL_PS,
                                          'tolerance': TOLERANCE, 'back': BACK, 'release': RELEASE,
                                          'release_iter': RELEASE_ITER, 'fmax': FMAX, 'max_step': MAX_STEP, 'probe': PROBE,
                                          'min_converged': MIN_CONVERGED, 'hessian': HESSIAN,
                                          'parallel': PARALLEL}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC con la GPU

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Estructura (PDB 2PTC)

In [ ]:
PDB = Path('/content/2PTC.pdb')
if not PDB.exists():
    run(f'wget -q -O {PDB} https://files.rcsb.org/download/2PTC.pdb')
print(sum(1 for l in PDB.read_text().splitlines() if l.startswith('ATOM')), 'átomos ATOM en 2PTC')

## 6. Prueba y mapas

Una carpeta por semilla (`seed_N/`): `scan.csv` (una fila por punto: objetivos y valores alcanzados de las dos
coordenadas, energía sin las restricciones, calor QM, fuerza más grande al final sobre un átomo móvil y sobre uno QM,
si convergió, frecuencias más bajas de las estructuras sin restricciones, distancias vigiladas, tiempo),
`scan_frames.pdb` (el sitio activo en cada punto) y `scan.log`. La prueba va en `prueba/seed_1/`. Lo terminado se
salta; una semilla interrumpida empieza de nuevo. Cada minuto se imprime la última línea de cada réplica.

In [ ]:
WATCH = 'E:195:OG-E:195:HG,E:195:HG-E:57:NE2,I:15:C-I:15:O,I:15:C-I:16:N,E:57:HD1-E:102:OD2'

def scans(base, seeds, step, step2, parallel, extra=''):
    # one qmmm_scan.py per seed, all at once on the GPU when parallel; returns {seed: exit code}
    todo = [s for s in seeds if not (base / f'seed_{s}' / 'scan.csv').exists()]
    if len(todo) < len(seeds):
        print('ya terminadas:', [s for s in seeds if s not in todo])
    procs = {}

    def report():
        for seed, (p, t0) in procs.items():
            lines = [l for l in (base / f'seed_{seed}' / 'scan.log').read_text().splitlines()
                     if l.strip() and not l.startswith('[MOZYME')]
            state = 'corriendo' if p.poll() is None else f'exit {p.returncode}'
            print(f'semilla {seed} ({state}, {(time.time() - t0) / 60:.0f} min): {lines[-1][:160] if lines else ""}',
                  flush=True)

    for seed in todo:
        d = base / f'seed_{seed}'
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PDB, d / PDB.name)
        cmd = (f'{sys.executable} {SRC}/scripts/qmmm/qmmm_scan.py {PDB.name} --libmopac {LIBMOPAC} '
               f'--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --scan {SCAN} --to {TO} '
               f'--step {step} --scan2 {SCAN2} --from2 {FROM2} --to2 {TO2} --step2 {step2} '
               f'--watch {WATCH} --tolerance {TOLERANCE} --fmax {FMAX} --max-step {MAX_STEP} --min-iter {MIN_ITER} '
               f'--min-iter-start {MIN_ITER_START} --mm-equil-ps {MM_EQUIL_PS} --seed {seed} --platform CUDA'
               + (' --back' if BACK else '') + (f' --release --release-iter {RELEASE_ITER}' if RELEASE else '')
               + (' --hessian' if HESSIAN else '') + extra)
        threads = max(1, NCPU // (len(todo) if parallel else 1))
        print('$', cmd, flush=True)
        procs[seed] = (subprocess.Popen(cmd, shell=True, executable='/bin/bash', cwd=d,
                                        stdout=open(d / 'scan.log', 'w'), stderr=subprocess.STDOUT,
                                        env=dict(os.environ, OMP_NUM_THREADS=str(threads))), time.time())
        while not parallel and procs[seed][0].poll() is None:
            time.sleep(60); report()
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(60); report()
    report()
    return {s: p.returncode for s, (p, _) in procs.items()}

if TEST_FIRST:
    TEST = OUT / 'prueba'
    rc = scans(TEST, [1], 0.4, 1.1, False, f' --probe {PROBE}' if PROBE else '')
    log = (TEST / 'seed_1' / 'scan.log').read_text() if (TEST / 'seed_1' / 'scan.log').exists() else ''
    ok = (TEST / 'seed_1' / 'scan.csv').exists() and ' r_g' in log and 'imaginary' in log
    print(f"\nprueba: {'bien' if ok else 'FALLÓ'}; códigos de salida {rc}")
    if not ok:
        print(log[-4000:])
        raise SystemExit('la prueba falló: no corro las réplicas')
    lines = [l for l in log.splitlines() if not l.startswith('[MOZYME')]
    print('\n'.join(l[:220] for l in lines if 'target' in l or 'probe' in l or 'along the force' in l
                     or 'no step kept' in l or 'SCF afresh' in l))
    trows = list(csv.DictReader(open(TEST / 'seed_1' / 'scan.csv')))
    frac = sum(float(r['conv']) for r in trows) / len(trows)
    print(f"\nprueba: convergieron {frac:.0%} de {len(trows)} minimizaciones")
    if frac < MIN_CONVERGED:
        raise SystemExit(f'menos de {MIN_CONVERGED:.0%} convergieron: no corro las réplicas')
rc = scans(OUT, SEEDS, STEP, STEP2, PARALLEL)
print('códigos de salida:', rc)

## 7. Resultados

Por réplica: el mapa (energía de cada punto contra los valores alcanzados de las dos coordenadas) con las
minimizaciones sin restricciones (flechas) y una tabla. Las energías se miden desde la referencia: la estructura sin
restricciones más baja de la región del reactivo (protón en Ser195, OG···C > 2 Å). El aducto es la estructura sin
restricciones más baja con OG···C < 1,7 Å y el protón en His57. El camino más bajo va entre puntos vecinos de la malla
(diagonales incluidas), desde el primer punto hasta alguno de los puntos desde los que la minimización sin restricciones
llegó al aducto, y es el que tiene el punto más alto más bajo. (Antes iba solo hasta el punto del que salió el aducto
más bajo; en 9a0c007a ese punto era a veces uno de la esquina, con el protón empujado por la restricción a 1,03 Å de
NE2, y el punto más alto del camino era entonces ese mismo punto forzado.) Ese punto más alto es un punto de la malla,
con restricciones: no es un estado de transición, y entre dos puntos vecinos puede haber más energía.

In [ ]:
import csv, heapq
import numpy as np
import matplotlib.pyplot as plt

def read(d):
    with open(d / 'scan.csv') as fh:
        rows = list(csv.DictReader(fh))
    for r in rows:
        for k in r:
            if k != 'window':
                r[k] = float(r[k])
    return rows

def lowest_path(g, start, goal):
    best, prev, pq = {start: g[start]['e']}, {}, [(g[start]['e'], start)]
    while pq:
        h, p = heapq.heappop(pq)
        if h > best[p]:
            continue
        for q in ((p[0] + a, p[1] + b) for a in (-1, 0, 1) for b in (-1, 0, 1) if a or b):
            if q in g and max(h, g[q]['e']) < best.get(q, np.inf):
                best[q], prev[q] = max(h, g[q]['e']), p
                heapq.heappush(pq, (best[q], q))
    path = [goal]
    while path[-1] != start:
        path.append(prev[path[-1]])
    return path[::-1]

kcal = 1 / 4.184
done = [s for s in SEEDS if (OUT / f'seed_{s}' / 'scan.csv').exists()]
fig, axs = plt.subplots(1, max(1, len(done)), figsize=(4.6 * max(1, len(done)), 4.0), squeeze=False)
summary = []
for ax, seed in zip(axs[0], done):
    rows = read(OUT / f'seed_{seed}')
    byname = {r['window']: r for r in rows}
    free = [r for r in rows if r['window'] == 'reactant' or r['window'].startswith('r_')]
    ref = min((r for r in free if r['d2'] < 0 and r['d'] > 2.0), key=lambda r: r['e'])
    e0 = ref['e']
    g = {tuple(int(v) for v in r['window'][1:].split('_')): r for r in rows if r['window'].startswith('g')}
    nc = [r['window'] for r in rows if not r['conv']]
    print(f"semilla {seed}: {len(rows)} minimizaciones, {len(nc)} sin converger"
          + (f" ({', '.join(nc[:10])}{' ...' if len(nc) > 10 else ''})" if nc else '')
          + f"; fuerza más grande al final: {max(r['fmax'] for r in rows):.1f} kJ/mol/nm sobre un átomo móvil, "
          f"{max(r['fmax_qm'] for r in rows):.1f} sobre uno QM; {sum(r['evals'] for r in rows):.0f} evaluaciones QM/MM, "
          f"{sum(r['scf_fail'] for r in rows):.0f} SCF sin converger y empezados de nuevo, "
          f"{sum(r.get('no_step', 0) for r in rows):.0f} búsquedas de línea sin paso")
    print(f"   referencia: {ref['window']} (OG–C {ref['d']:.2f} Å, protón {ref['d2']:+.2f} Å); la estructura de "
          f"partida está {(rows[0]['e'] - e0) * kcal:+.1f} kcal/mol por encima")
    for r in free:
        s0 = byname.get(r['window'][2:], r)
        fr = '' if np.isnan(r['n_imag']) else f"; imaginarias {r['n_imag']:.0f}, la más baja {r['freq1']:.0f} cm-1"
        print(f"   {r['window']:>9s}: desde OG–C {s0['d']:.2f}, protón {s0['d2']:+.2f} -> OG–C {r['d']:.2f} Å, protón "
              f"{r['d2']:+.2f} Å, {(r['e'] - e0) * kcal:+6.1f} kcal/mol, fuerza QM {r['fmax_qm']:.1f}{fr}")
    x = np.array([r['d'] for r in g.values()]); y = np.array([r['d2'] for r in g.values()])
    z = np.array([(r['e'] - e0) * kcal for r in g.values()])
    cs = ax.tricontourf(x, y, z, levels=20, cmap='viridis')
    ax.plot(x, y, '.', color='w', ms=2)
    fig.colorbar(cs, ax=ax, label='energía − referencia (kcal/mol)')
    for r in free:
        if r['window'].startswith('r_'):
            s0 = byname[r['window'][2:]]
            ax.annotate('', xy=(r['d'], r['d2']), xytext=(s0['d'], s0['d2']),
                        arrowprops=dict(arrowstyle='->', color='r', lw=1))
    ax.plot(ref['d'], ref['d2'], '*', color='r', ms=12, label='referencia')
    add = [r for r in free if r['d'] < 1.7 and r['d2'] > 0 and r['conv']]
    if add:
        ad = min(add, key=lambda r: r['e'])
        # the grid points from which the minimization without restraints reached the adduct basin
        reached = {r['window'] for r in add}
        goals = [k for k, r in g.items() if 'r_' + r['window'] in reached]
        paths = [lowest_path(g, (0, 0), k) for k in goals]
        path = min(paths, key=lambda pa: max(g[q]['e'] for q in pa))
        qt = max(path, key=lambda q: g[q]['e'])
        top = (g[qt]['e'] - e0) * kcal
        ax.plot([g[q]['d'] for q in path], [g[q]['d2'] for q in path], '-', color='orange', lw=1.5,
                label='camino más bajo')
        print(f"   aducto: {ad['window']} {(ad['e'] - e0) * kcal:+.1f} kcal/mol (OG–C {ad['d']:.2f} Å, C–O "
              f"{ad['I:15:C-I:15:O']:.3f}, C–N {ad['I:15:C-I:16:N']:.3f} Å, imaginarias {ad['n_imag']:.0f}; "
              f"{len(add)} minimizaciones sin restricciones llegaron a él, entre "
              f"{(min(r['e'] for r in add) - e0) * kcal:+.2f} y {(max(r['e'] for r in add) - e0) * kcal:+.2f}); "
              f"punto más alto del camino más bajo {top:+.1f} kcal/mol ({g[qt]['window']}, OG–C {g[qt]['d']:.2f} Å, "
              f"protón {g[qt]['d2']:+.2f} Å), {top - (ad['e'] - e0) * kcal:.1f} por encima del aducto; "
              f"{sum(g[q]['conv'] for q in path):.0f} de sus {len(path)} puntos convergieron")
        summary.append(((ad['e'] - e0) * kcal, top))
    else:
        print('   ninguna estructura sin restricciones quedó en el aducto (OG–C < 1,7 Å, protón en His57)')
    ida = {(round(r['target'], 3), round(r['target2'], 3)): r for r in g.values()}
    dif = [(r['e'] - ida[k]['e']) * kcal for r in rows if r['window'].startswith('b')
           for k in [(round(r['target'], 3), round(r['target2'], 3))] if k in ida]
    if dif:
        print(f"   vuelta − ida en {len(dif)} puntos: de {min(dif):+.2f} a {max(dif):+.2f} kcal/mol")
    print(f"   tiempo: {sum(r['seconds'] for r in rows) / 60:.0f} min de minimización")
    ax.invert_xaxis()
    ax.set_xlabel('Ser195 OG – Lys15 C (Å)'); ax.set_ylabel('d(OG–HG) − d(HG–NE2) (Å)')
    ax.set_title(f'semilla {seed}'); ax.legend(loc='upper left', fontsize=7)
if len(summary) > 1:
    a_ = np.array([v[0] for v in summary]); t_ = np.array([v[1] for v in summary])
    n_ = len(summary)
    print(f"\n{n_} réplicas (media ± desviación estándar; error estándar de la media): aducto {a_.mean():.1f} ± "
          f"{a_.std(ddof=1):.1f} kcal/mol ({a_.std(ddof=1) / n_ ** 0.5:.1f}; de {a_.min():.1f} a {a_.max():.1f}); "
          f"punto más alto del camino {t_.mean():.1f} ± {t_.std(ddof=1):.1f} ({t_.std(ddof=1) / n_ ** 0.5:.1f}; de "
          f"{t_.min():.1f} a {t_.max():.1f}); punto más alto − aducto de {(t_ - a_).min():.1f} a {(t_ - a_).max():.1f}")
# reproducibility: the short test and replica 1 share the seed, the MM equilibration and the reactant minimization
tl = OUT / 'prueba' / 'seed_1' / 'scan.log'
if tl.exists() and (OUT / 'seed_1' / 'scan.log').exists():
    pick = lambda t: [l for l in t.splitlines() if l.startswith(('system:', 'MM minimization'))
                      or l.lstrip().startswith('reactant target')]
    a1, b1 = pick(tl.read_text()), pick((OUT / 'seed_1' / 'scan.log').read_text())
    print('\nreproducibilidad (prueba y réplica 1, la misma semilla):')
    for x1, y1 in zip(a1, b1):
        print('  prueba:   ', x1[:150]); print('  réplica 1:', y1[:150])
fig.tight_layout()
fig.savefig(OUT / 'enzyme_scan.png', dpi=150)
plt.show()

## 8. Descargar

In [ ]:
tmp = Path('/content/_enzyme') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('env.json', 'commit.txt', 'enzyme_scan.png'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for d in sorted(OUT.glob('seed_*')) + sorted(OUT.glob('prueba/seed_*')):
    (tmp / d.relative_to(OUT)).mkdir(parents=True)
    for f in ('scan.csv', 'scan_frames.pdb', 'scan.log', 'scan_probe.csv'):
        if (d / f).exists():
            shutil.copy2(d / f, tmp / d.relative_to(OUT) / f)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)